# Build Feature Dataset and Train/Test Split

The `dataset-training-model` file committed at the repo root expects `good_repos.csv` /
`uncertain.csv` to already carry feature columns (stars, forks, etc.), but those files only have a
`repo` column. This notebook builds the missing feature-enriched dataset by fetching live GitHub
data, then runs the same splitting approach as that file (random seed 42, 80/20 stratified split,
no-repo-overlap check) against real data so it actually produces `data/train_split.csv` and
`data/test_split.csv`.

Features computed per repo, all from Tasks 1 and 3's signal work:

- Raw counts: `stargazers_count`, `forks_count`, `subscribers_count`, `open_issues_count`, `size`
- Derived: `age_days`, `days_since_push`, `fork_star_ratio`, `subscriber_star_ratio`,
  `issues_per_star`, `stars_per_day`, `has_license`, `is_org_owned`
- `contributor_activity_ratio` and `ghost_contributor_ratio` (`src/gh_signals/signals.py`)

`star_growth_burstiness` is left out: GitHub restricted the stargazer-listing endpoint in July
2026 to admins/collaborators, so it cannot be computed for a repo we do not own (see the
docstring on `GitHubClient.get_stargazer_timestamps`).

In [1]:
import asyncio
import csv
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

from gh_signals.github_client import GitHubClient, GitHubError
from gh_signals.signals import contributor_activity_ratio, ghost_contributor_ratio, is_bot_contributor

DATA_DIR = Path("..", "data")
TOP_N = 10
RANDOM_SEED = 42

## Fetch features for every repo

One `get_repo` call, one `get_contributor_stats` call, and up to `TOP_N` `get_user` calls per
repo (for the ghost-contributor check). Bounded concurrency keeps this from hammering the API.

In [2]:
def load_repos(csv_path: Path) -> list[str]:
    with csv_path.open() as file:
        return [row["repo"] for row in csv.DictReader(file)]


good_repos = load_repos(DATA_DIR / "good_repos.csv")
uncertain_repos = load_repos(DATA_DIR / "uncertain.csv")
all_repos = [(repo, "good") for repo in good_repos] + [(repo, "uncertain") for repo in uncertain_repos]
len(all_repos)

106

In [3]:
async def fetch_repo_row(client: GitHubClient, repo: str, label: str) -> dict:
    repo_info = await client.get_repo(repo)
    stats = await client.get_contributor_stats(repo)

    candidates = [c for c in stats if c.author is not None and not is_bot_contributor(c)]
    top = sorted(candidates, key=lambda c: c.total, reverse=True)[:TOP_N]
    users = []
    for contributor in top:
        if contributor.author is None:
            continue
        try:
            users.append(await client.get_user(contributor.author.login))
        except GitHubError:
            continue

    now = datetime.now(UTC)
    age_days = (now - repo_info.created_at).days
    stars = repo_info.stargazers_count

    return {
        "full_name": repo_info.full_name,
        "label": label,
        "stargazers_count": stars,
        "forks_count": repo_info.forks_count,
        "subscribers_count": repo_info.subscribers_count,
        "open_issues_count": repo_info.open_issues_count,
        "size": repo_info.size,
        "age_days": age_days,
        "days_since_push": (now - repo_info.pushed_at).days,
        "fork_star_ratio": repo_info.forks_count / stars if stars else None,
        "subscriber_star_ratio": repo_info.subscribers_count / stars if stars else None,
        "issues_per_star": repo_info.open_issues_count / stars if stars else None,
        "stars_per_day": stars / max(age_days, 1),
        "has_license": repo_info.license is not None,
        "is_org_owned": repo_info.owner.type == "Organization",
        "contributor_activity_ratio": contributor_activity_ratio(stats),
        "ghost_contributor_ratio": ghost_contributor_ratio(users),
    }


fetch_sem = asyncio.Semaphore(8)


async def fetch_bounded(client: GitHubClient, repo: str, label: str) -> dict:
    async with fetch_sem:
        return await fetch_repo_row(client, repo, label)


async with GitHubClient() as client:
    rows = await asyncio.gather(*(fetch_bounded(client, repo, label) for repo, label in all_repos))

df = pd.DataFrame(rows)
df.to_csv(DATA_DIR / "repo_features.csv", index=False)
df.head()

,full_name,label,stargazers_count,forks_count,subscribers_count,open_issues_count,size,age_days,days_since_push,fork_star_ratio,subscriber_star_ratio,issues_per_star,stars_per_day,has_license,is_org_owned,contributor_activity_ratio,ghost_contributor_ratio
0,angular/angular,good,101027,29047,2999,1141,659783,4396,0,0.287517,0.029685,0.011294,22.981574,True,True,0.086519,0.0
1,anthropics/skills,good,179341,21204,1125,1392,5170,374,3,0.118233,0.006273,0.007762,479.521390,False,True,0.375000,0.0
2,ansible/ansible,good,70822,24337,1912,865,266273,5322,1,0.343636,0.026997,0.012214,13.307403,True,True,0.024000,0.0
3,apache/airflow,good,47025,17935,794,1825,734728,4189,0,0.381393,0.016885,0.038809,11.225830,True,True,0.234940,0.0
4,apache/spark,good,44107,29399,1998,590,617825,4601,0,0.666538,0.045299,0.013377,9.586394,True,True,0.166000,0.0


`data/repo_features.csv` is the feature-enriched replacement for what `good_repos.csv` /
`uncertain.csv` would need to contain for the committed split script to work. It is the input to
the split below, and to Part 2's per-signal evaluation in
`notebooks/003_signal_auc_evaluation.ipynb`.

## Data integrity checks

Same checks called for in the task: duplicate repos, missing labels, data types, and missing
feature values per column (documented here rather than silently dropped, since how a model should
handle a missing value is a modeling decision, not a data-prep one).

In [4]:
print("Data types:")
print(df.dtypes)
print()
print("Duplicate full_name rows:", df["full_name"].duplicated().sum())
print("Rows missing label:", df["label"].isna().sum())
print()
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

Data types:
full_name                         str
label                             str
stargazers_count                int64
forks_count                     int64
subscribers_count               int64
open_issues_count               int64
size                            int64
age_days                        int64
days_since_push                 int64
fork_star_ratio               float64
subscriber_star_ratio         float64
issues_per_star               float64
stars_per_day                 float64
has_license                      bool
is_org_owned                     bool
contributor_activity_ratio    float64
ghost_contributor_ratio       float64
dtype: object

Duplicate full_name rows: 0
Rows missing label: 0

Missing values per column:
Series([], dtype: int64)


No duplicates and no missing labels. Every repo in this dataset has at least one star, so the
`*_per_star` ratios never divide by zero; the only columns that can be missing are
`contributor_activity_ratio` and `ghost_contributor_ratio`, for a repo with zero checkable human
contributors. None of the 106 repos hit that case on this run (see
`notebooks/004_ghost_contributor_detection.ipynb` for that edge case being checked explicitly),
but a model trained on this data should still impute or otherwise handle a `NaN` in those two
columns rather than assume they are always populated.

## Train/test split

Same approach as the committed `dataset-training-model` script: drop duplicate repos, drop rows
missing the label, exclude identifier and target-revealing columns from the feature set, and split
80/20 with a fixed random seed, stratified on the label so both classes keep their original
proportion in each split.

In [5]:
df["target_label"] = (df["label"] == "good").astype(int)
df = df.drop_duplicates(subset=["full_name"]).dropna(subset=["target_label"])

EXCLUDE_COLS = ["full_name", "label", "target_label"]
feature_cols = [col for col in df.columns if col not in EXCLUDE_COLS]

X = df[feature_cols]
y = df["target_label"]

X_train, X_test, y_train, y_test, train_df, test_df = train_test_split(
    X,
    y,
    df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

print(f"Total repositories: {len(df)}")
print(f"Training set: {len(train_df)} ({len(train_df) / len(df):.1%})")
print(f"Testing set:  {len(test_df)} ({len(test_df) / len(df):.1%})")
print()
print("Label distribution (1 = good, 0 = uncertain):")
print("Full dataset:\n", df["target_label"].value_counts(normalize=True))
print("Training set:\n", train_df["target_label"].value_counts(normalize=True))
print("Testing set:\n", test_df["target_label"].value_counts(normalize=True))

overlap = set(train_df["full_name"]).intersection(set(test_df["full_name"]))
assert len(overlap) == 0, f"Overlap detected: {overlap}"
print("\nNo repository overlap between train and test: confirmed")

train_df.to_csv(DATA_DIR / "train_split.csv", index=False)
test_df.to_csv(DATA_DIR / "test_split.csv", index=False)

Total repositories: 106
Training set: 84 (79.2%)
Testing set:  22 (20.8%)

Label distribution (1 = good, 0 = uncertain):
Full dataset:
 target_label
1    0.773585
0    0.226415
Name: proportion, dtype: float64
Training set:
 target_label
1    0.77381
0    0.22619
Name: proportion, dtype: float64
Testing set:
 target_label
1    0.772727
0    0.227273
Name: proportion, dtype: float64

No repository overlap between train and test: confirmed


`data/train_split.csv` and `data/test_split.csv` are now real, saved files: 84 repos train /
22 test (80/20 of 106, stratified so both splits keep roughly the same 82:24 good:uncertain
ratio as the full dataset), seed 42, verified no repository overlap. `notebooks/003_signal_auc_evaluation.ipynb` picks up from here for Part 2.